In [ ]:
%%capture
!pip install --upgrade scikit-learn
!pip install --upgrade gensim
!apt-get install git

In [ ]:
%%capture
!git clone https://github.com/YJiangcm/SST-2-sentiment-analysis.git

In [ ]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np

import pandas as pd

def Preprocess(Text):
    msg = re.sub('[^a-zA-Z]', ' ', Text)
    msg = msg.lower()
    msg = msg.split()
    msg = ' '.join(msg)
    return msg

In [ ]:
#FTX = gensim.downloader.load('fasttext-wiki-news-subwords-300')

In [ ]:
trainset = pd.read_csv("SST-2-sentiment-analysis/data/train.tsv", sep='\t', header=None, names=["class", "review"])
testset = pd.read_csv("SST-2-sentiment-analysis/data/test.tsv", sep='\t', header=None, names=["class", "review"])
trainset["p-review"] = trainset["review"].apply(Preprocess)
testset["p-review"] = testset["review"].apply(Preprocess)

In [ ]:
def splitClsData(dset):
  POS=[]
  NEG=[]
  VOCAB = {"<pad>": 0, "<s>": 1, "</s>": 2, "<unk>": 3}
  nVOCAB = 4
  for idx in range(len(dset)):
    words = dset["p-review"][idx].split()
    for word in words:
      if VOCAB.get(word) == None:
        VOCAB[word] = nVOCAB
        nVOCAB += 1
    words = ['<s>'] + words + ['</s>']
    if trainset["class"][idx] == 0:
      POS.append(words)
    else:
      NEG.append(words)
  return POS, NEG, VOCAB

def tokenize(words, VOCAB):
  ids = []
  for word in words:
    if VOCAB.get(word) == None:
      ids.append(VOCAB["<unk>"])
    else:
      ids.append(VOCAB[word])
  return ids

def batchSamples(samples, VOCAB, bsize=16):
  batches = []
  for i in range(len(samples) // bsize):
    maxlen = 0
    for j in range(bsize):
      l = len(samples[i*bsize + j])
      if maxlen < l:
        maxlen = l
    batch = np.zeros((bsize, maxlen))
    for j in range(bsize):
      l = len(samples[i*bsize + j])
      ids = samples[i*bsize + j]
      if l < maxlen:
         ids += [VOCAB["<pad>"]] * (maxlen - l)
      batch[j] = ids
    batches.append(batch)
  return batches

In [ ]:
class LSTMLanguageModel(nn.Module):

    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(vocab_size, embed_size)

        self.Wi = nn.Linear(embed_size, hidden_size)
        self.Wf = nn.Linear(embed_size, hidden_size)
        self.Wo = nn.Linear(embed_size, hidden_size)
        self.Wg = nn.Linear(embed_size, hidden_size)

        self.Ui = nn.Linear(hidden_size, hidden_size)
        self.Uf = nn.Linear(hidden_size, hidden_size)
        self.Uo = nn.Linear(hidden_size, hidden_size)
        self.Ug = nn.Linear(hidden_size, hidden_size)

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, state=None):

        batch_size, seq_len = x.shape

        if state is None:
            h = torch.zeros(batch_size, self.hidden_size, device=x.device)
            c = torch.zeros(batch_size, self.hidden_size, device=x.device)
        else:
            h, c = state

        x = self.embedding(x)

        outputs = []

        for t in range(seq_len):

            xt = x[:, t, :]

            i = torch.sigmoid(self.Wi(xt) + self.Ui(h))
            f = torch.sigmoid(self.Wf(xt) + self.Uf(h))
            o = torch.sigmoid(self.Wo(xt) + self.Uo(h))
            g = torch.tanh(self.Wg(xt) + self.Ug(h))

            c = f * c + i * g
            h = o * torch.tanh(c)

            out = self.fc(h)

            outputs.append(out)

        outputs = torch.stack(outputs, dim=1)

        return outputs, (h, c)

In [ ]:
#dim = FTX.vector_size

POSSents, NEGSents, VOCAB = splitClsData(trainset)

LOOKUP = {}
for word in VOCAB:
  LOOKUP[VOCAB[word]] = word



POS = []
NEG = []

for sent in POSSents:
  POS.append(tokenize(sent, VOCAB))

for sent in NEGSents:
  NEG.append(tokenize(sent, VOCAB))

POSBatches = batchSamples(POS, VOCAB)
NEGBatches = batchSamples(NEG, VOCAB)

In [ ]:
device = "cuda"

vocab_size = len(VOCAB)
model = LSTMLanguageModel(vocab_size, 256, 256).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

for epoch in range(20):

    total_loss = 0

    for batch in POSBatches:

        x = torch.LongTensor(batch[:,:-1]).to(device)
        y = torch.LongTensor(batch[:,1:]).to(device)

        logits, _ = model(x)

        loss = criterion(
            logits.view(-1, vocab_size),
            y.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("epoch", epoch, total_loss)

epoch 0 1415.3243460655212
epoch 1 1173.827437877655
epoch 2 990.9779238700867
epoch 3 811.211314201355
epoch 4 666.8431751728058
epoch 5 549.3636540174484
epoch 6 456.4288184642792
epoch 7 381.5056309700012
epoch 8 318.0446183681488
epoch 9 269.1142592430115
epoch 10 232.6571227312088
epoch 11 203.4862968325615
epoch 12 182.47114253044128
epoch 13 164.78331220149994
epoch 14 153.76927906274796
epoch 15 147.06301671266556
epoch 16 144.22321325540543
epoch 17 139.12604784965515
epoch 18 134.67319828271866
epoch 19 132.63857650756836


In [ ]:
import random

def selectID(arr, topk=50):
  sel = random.random()
  toparr = sorted(((val, idx) for idx, val in enumerate(arr)),reverse=True)[:topk]
  ex = [p for p, _ in toparr]
  ex = np.exp(ex)
  prob = ex/np.sum(ex)
  b = 0
  nwords = len(toparr)
  for i in range(nwords):
    b += prob[i]
    prob[i] = b
  b = 0
  id = 0
  for i in range(nwords):
    if sel > b and sel <= prob[i]:
      id = i
      break;
    b = prob[i]
  return toparr[id][1]

def generate(model, VOCAB, LOOKUP, topk=50):
  genText = []
  sents = []

  vocabSize = len(VOCAB)
  id = VOCAB["<s>"]

  x = torch.LongTensor([[id]]).to(device)

  out, h = model(x)

  id = selectID(out[0].detach().cpu().numpy()[0], topk=topk)
  predWord = LOOKUP[id]
  genText.append(predWord)
  if predWord == "</s>":
    return genText
  predX = torch.LongTensor([[id]]).to(device)
  for i in range(80):
    out, h = model(predX, h)
    id = selectID(out[0].detach().cpu().numpy()[0], topk=topk)
    predWord = LOOKUP[id]
    predX = torch.LongTensor([[id]]).to(device)
    if predWord == "</s>":
      return genText
    genText.append(predWord)

  return genText

In [ ]:
for i in range(20):
  words = generate(model, VOCAB, LOOKUP)
  print(" ".join(words))

it s hard to imagine any recent film independent or otherwise that makes as much of a mess as this one
it s a bad action movie because there s no rooting interest and the spectacle is grotesque and boring
the characters are based on stock clich s and the attempt to complicate the story only defies credibility
it all comes down to whether you can tolerate leon barlow
it s hard to understand why anyone in his right mind would even make to the level of classic
feels like a change in which murder by concentrating things away all the queen s men and tiresome project cut which adds characters for the absence of narrative continuity undisputed is nearly incoherent an excuse to get to the closing bout by which time it s impossible to care who wins
a zombie movie in every sense is bad something happens and annoying
is that it s inauthentic at being one
the story is bogus and its characters tissue thin
it s harder still to believe that anyone in his right mind would even think we are likely to b